# Lesson 05 — Cleaning the Foreground Mask with Morphology

```python
import cv2
import numpy as np
import matplotlib.pyplot as plt

cap = cv2.VideoCapture('sample_video.mp4')
mog2 = cv2.createBackgroundSubtractorMOG2(history=300, varThreshold=50, detectShadows=False)
for _ in range(30): cap.read()
ret, frame = cap.read()
raw_mask = mog2.apply(frame)
cap.release()

# Progressive cleaning pipeline
se_small = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3,3))
se_med   = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7,7))
se_large = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(15,15))

step1 = cv2.morphologyEx(raw_mask, cv2.MORPH_OPEN,  se_small)  # remove tiny noise
step2 = cv2.morphologyEx(step1,    cv2.MORPH_CLOSE, se_med)    # fill holes in blobs
step3 = cv2.dilate(step2, se_large, iterations=1)              # expand to cover full object

fig, axes = plt.subplots(1,4,figsize=(22,5))
for ax, im, t in zip(axes,
    [raw_mask, step1, step2, step3],
    ['Raw MOG2 mask','After OPEN (noise gone)','After CLOSE (holes filled)','After DILATE (expanded)']):
    ax.imshow(im, cmap='gray'); ax.set_title(t); ax.axis('off')
plt.suptitle('Morphological cleaning pipeline for foreground masks',fontsize=12)
plt.show()

# Final: find contours on clean mask
contours, _ = cv2.findContours(step3, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
valid = [c for c in contours if cv2.contourArea(c) > 1000]
vis = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB) if ret else np.zeros((480,640,3),dtype=np.uint8)
print(f"Clean objects detected: {len(valid)}")